In [1]:
import torch
print("Using torch", torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using torch 2.0.1+cu118
Using device: cuda


In [2]:
import numpy as np
from tqdm import tqdm

n_alpha=36
n_beta=36
n_insert=0
alpha_array=np.linspace(0,90,n_alpha+1)[:-1]
beta_array=np.linspace(0,180,n_beta+1)[:-1]

flat_x=np.zeros((n_alpha*(n_insert+1),n_beta*(n_insert+1),40*40),dtype=np.float32)
flat_y=np.zeros((n_alpha*(n_insert+1),n_beta*(n_insert+1),3),dtype=np.float32)

for i in tqdm(range(n_alpha)):
    for j in range(n_beta):
        cur_file=np.load("./training_set_processed/alpha%.2f_beta%.2f.npz"%(alpha_array[i],beta_array[j]))
        flat_x[i*(n_insert+1):(i+1)*(n_insert+1),j*(n_insert+1):(j+1)*(n_insert+1)]=cur_file["arr_0"]
        flat_y[i*(n_insert+1):(i+1)*(n_insert+1),j*(n_insert+1):(j+1)*(n_insert+1)]=cur_file["arr_1"]
flat_x=flat_x.reshape(-1,40*40)
flat_y=flat_y.reshape(-1,3)

x_train=torch.from_numpy(flat_x.reshape(-1,1,40,40)).to(device)
y_train=torch.from_numpy(flat_y).to(device)

100%|██████████████████████████████████████████████████████████████████████████████████| 36/36 [00:17<00:00,  2.01it/s]


In [3]:
fo=np.load("./val_set/data_processed.npz")
x_val=torch.from_numpy(fo["arr_0"].reshape(-1,1,40,40)).to(device)
y_val=torch.from_numpy(fo["arr_1"]).to(device)

fo=np.load("./test_set/data_processed.npz")
x_test=torch.from_numpy(fo["arr_0"].reshape(-1,1,40,40)).to(device)
y_test=torch.from_numpy(fo["arr_1"]).to(device)

In [4]:
import torch.nn as nn

class Simple_CNN(nn.Module):
    def __init__(self,p):
        super().__init__()
        self.conv1 = nn.Sequential(         
            nn.Conv2d(1,16,5,1,2),                              
            nn.ReLU(),            
            nn.MaxPool2d(kernel_size=2),    
        )
        self.conv2 = nn.Sequential(         
            nn.Conv2d(16,32,3,1,1),     
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),                
        )
        self.conv3 = nn.Sequential(         
            nn.Conv2d(32,64,3,1,1),     
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),                
        )
        self.mlp = nn.Sequential(
            nn.Dropout(p=p),
            nn.Linear(5*5*64,1024),
            nn.ReLU(),
            nn.Linear(1024,1024),
            nn.ReLU(),
            nn.Linear(1024,3),
        )
        
    def forward(self, x):
        y = self.conv1(x)
        y = self.conv2(y)
        y = self.conv3(y)
        y = y.view(y.size(0), -1)       
        y = self.mlp(y)
        return y

In [8]:
def train(model,X,y,optimizer,loss_fn,batchsize=650):
    idx=np.arange(X.shape[0])
    np.random.shuffle(idx)
    
    MAE = 0
    epoch_loss = 0
    model.train()
    for i in range(0,X.shape[0],batchsize):
        if i+batchsize>=X.shape[0]:
            cur_idx=idx[i:]
        else:
            cur_idx=idx[i:i+batchsize]
            
        optimizer.zero_grad()
        y_pred = model(X[cur_idx])
        MAE = MAE + torch.sum(torch.abs(y_pred.detach()-y[cur_idx])).item()
        loss = loss_fn(y_pred, y[cur_idx])
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()*cur_idx.shape[0]
    
    MAE = MAE/y.shape[0]/3
    
    return epoch_loss/X.shape[0],MAE
    
def evaluate(model,X,y,loss_fn,batchsize=5000):
    epoch_loss = 0
    separate_loss=np.zeros(3,dtype=np.float32)
    model.eval()
    with torch.no_grad():
        for i in range(0,X.shape[0],batchsize):
            i_start=i
            if i+batchsize>=X.shape[0]:
                i_end=X.shape[0]-1
            else:
                i_end=i+batchsize
                
            y_pred=model(X[i_start:i_end])
            loss = loss_fn(y_pred, y[i_start:i_end])
            epoch_loss += loss.item()*(i_end-i_start)
            separate_loss += torch.sum(torch.abs(y_pred-y[i_start:i_end]),dim=0).to("cpu").numpy()
            
        epoch_loss=epoch_loss/X.shape[0]
        separate_loss=separate_loss/X.shape[0]
    return epoch_loss,separate_loss


In [9]:
from torch.nn import MSELoss
from torch.optim import Adam
from tqdm import tqdm 
import pandas as pd

def run_cnn_exp(task_name,p,ratio_list,
                    seed=1222,n_epoch=4000,save_dir="./ML_result/"):
    score=np.zeros((n_epoch,3))
    torch.manual_seed(seed)
    np.random.seed(seed)
    idx = np.arange(x_train.shape[0])
    np.random.shuffle(idx)
    
    for j in range(len(ratio_list)):
        print("Running %s %.2f"%(task_name,ratio_list[j]))
        end_idx = round(x_train.shape[0]*train_ratio[j])-1
        
        model=Simple_CNN(p)
        model = model.to(device)
        loss_fn=MSELoss()
        optimizer=Adam(model.parameters(),lr=0.001)
        result=np.zeros((n_epoch,7),dtype=np.float32)
        
        for i in tqdm(range(n_epoch)):
            result[i,0],result[i,1]=train(model,x_train[idx[:end_idx]],y_train[idx[:end_idx]],optimizer,loss_fn)
            result[i,2],_=evaluate(model,x_val,y_val,loss_fn)
            result[i,3],result[i,4:]=evaluate(model,x_test,y_test,loss_fn)
        
        best_idx=np.argmin(result[:,2])
        print(result[best_idx,4:])
        score[j] = result[best_idx,4:]
        result=pd.DataFrame(result,columns=["train_loss","train_MAE","validation_loss","test_loss","S1_MAE","S2_MAE","S3_MAE"])
        save_filename=save_dir+task_name+"_ratio%.2f"%ratio_list[j]+".csv"
        result.to_csv(save_filename,index=False)
        
    return score

In [10]:
train_ratio = [0.01,0.02,0.04,0.07,0.1,
               0.15,0.2,0.3,0.4,0.6,0.8,1]
run_cnn_exp("change_size",0.001,train_ratio)

Running change_size 0.01


100%|█████████████████████████████████████████████████████████████████████████████| 4000/4000 [00:27<00:00, 146.76it/s]


[0.134418   0.13629791 0.228783  ]
Running change_size 0.02


100%|█████████████████████████████████████████████████████████████████████████████| 4000/4000 [00:29<00:00, 136.52it/s]


[0.13978823 0.11438248 0.10404706]
Running change_size 0.04


...

100%|██████████████████████████████████████████████████████████████████████████████| 4000/4000 [02:56<00:00, 22.71it/s]


[0.01053097 0.00986765 0.01105187]
Running change_size 1.00


100%|██████████████████████████████████████████████████████████████████████████████| 4000/4000 [03:27<00:00, 19.24it/s]

[0.01111334 0.01160341 0.00851512]


array([[0.134418  , 0.13629791, 0.228783  ],
       [0.13978823, 0.11438248, 0.10404706],
       [0.0748098 , 0.06029357, 0.05319748],
       ...,
       [0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        ]])